# Part 1 on real MNIST: does the Lie bracket predict forgetting?

Run the cells **top to bottom, one at a time**. Each cell says what it should print.
If a cell prints something different or shows an error, stop and copy the output to Claude.

Same experiment as the pilot, now on real MNIST (28×28 digits, 60,000 training images):
- 5 tasks: 0 vs 1, 2 vs 3, 4 vs 5, 6 vs 7, 8 vs 9
- Learn task A, then task B. Measure how much A was forgotten, on test images.
- Every ordered pair (20) × several seeds, in two settings: **task** (mild forgetting) and **domain** (strong forgetting)
- Compare warning signals: Lie bracket (at the switch, and over the first 20 steps of B), EWC-style curvature, gradient size, gradient alignment, and the simple check (old task's training loss after 20 steps)

## Cell 1: settings
Change `BASE` if your hard drive has a different letter. Everything (data, results, figures) is saved there.

**Should print:** the folder paths.

In [2]:
import os
BASE = r"F:\forgetting"          # <-- change the drive letter if needed
DATA_DIR = os.path.join(BASE, "data")
RESULTS_DIR = os.path.join(BASE, "results")
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(RESULTS_DIR, exist_ok=True)

SEEDS = 5          # 5 matches the pilot. Use 3 if you're short on time.
EPOCHS = 2         # passes over each task's training data
LR = 0.05          # plain SGD learning rate
BATCH = 64
HIDDEN = 256       # MLP: 784 -> 256 -> 256 -> outputs
EST_SIZE = 2000    # training images per task used to measure gradients and the bracket
print("data:", DATA_DIR); print("results:", RESULTS_DIR)

data: F:\forgetting\data
results: F:\forgetting\results


## Cell 2: check the GPU
**Should print:** `cuda` and `NVIDIA GeForce RTX 3090`. If it says `cpu`, stop: PyTorch was installed without GPU support.

In [3]:
import time, json, csv
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.func import functional_call
from scipy.stats import spearmanr, rankdata
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", dev)
if dev.type == "cuda": print(torch.cuda.get_device_name(0))

torch 2.5.1+cu121 | device: cuda
NVIDIA GeForce RTX 3090


## Cell 3: get MNIST
The first time, this downloads MNIST (about 12 MB) into your data folder. After that it loads from disk.

**Should print:** `train 60000 test 10000`, then 5 lines with roughly 12,000 train / 2,000 test images per task.

If the download fails (no internet), see the guide: put the four MNIST `.gz` files in `data\MNIST\raw\` and run again.

In [4]:
from torchvision import datasets
tr = datasets.MNIST(DATA_DIR, train=True, download=True)
te = datasets.MNIST(DATA_DIR, train=False, download=True)
def prep(ds):
    X = ds.data.float().div(255.).view(-1, 784)
    return (X - 0.1307) / 0.3081, ds.targets.clone()
Xtr_all, ytr_all = prep(tr); Xte_all, yte_all = prep(te)
print("train", len(ytr_all), "test", len(yte_all))

TASKS = [(0, 1), (2, 3), (4, 5), (6, 7), (8, 9)]
NAMES = ["0 vs 1", "2 vs 3", "4 vs 5", "6 vs 7", "8 vs 9"]
data = {}
g0 = torch.Generator().manual_seed(123)
for t, cls in enumerate(TASKS):
    mtr = (ytr_all == cls[0]) | (ytr_all == cls[1]); mte = (yte_all == cls[0]) | (yte_all == cls[1])
    Xtr, ytr, Xte, yte = Xtr_all[mtr], ytr_all[mtr], Xte_all[mte], yte_all[mte]
    est = torch.randperm(len(ytr), generator=g0)[:EST_SIZE]
    data[t] = dict(Xtr=Xtr.to(dev), ytr=ytr.to(dev), Xte=Xte.to(dev), yte=yte.to(dev),
                   Xest=Xtr[est].to(dev), yest=ytr[est].to(dev), cls=torch.tensor(cls, device=dev))
    print(f"task {t} ({NAMES[t]}): train {len(ytr)}, test {len(yte)}")

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 9.91M/9.91M [00:22<00:00, 444kB/s] 


Extracting F:\forgetting\data\MNIST\raw\train-images-idx3-ubyte.gz to F:\forgetting\data\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 28.9k/28.9k [00:00<00:00, 121kB/s]


Extracting F:\forgetting\data\MNIST\raw\train-labels-idx1-ubyte.gz to F:\forgetting\data\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 1.65M/1.65M [00:01<00:00, 889kB/s] 


Extracting F:\forgetting\data\MNIST\raw\t10k-images-idx3-ubyte.gz to F:\forgetting\data\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 4.54k/4.54k [00:00<00:00, 1.39MB/s]


Extracting F:\forgetting\data\MNIST\raw\t10k-labels-idx1-ubyte.gz to F:\forgetting\data\MNIST\raw

train 60000 test 10000
task 0 (0 vs 1): train 12665, test 2115
task 1 (2 vs 3): train 12089, test 2042
task 2 (4 vs 5): train 11263, test 1874
task 3 (6 vs 7): train 12183, test 1986
task 4 (8 vs 9): train 11800, test 1983


## Cell 4: the model and the maths
Defines the small network, training (plain SGD), the Hessian-vector product, and the Lie bracket.

**Should print:** `ready`. Nothing runs yet.

In [5]:
SCENARIO = "task"   # set again before each run

class MLP(nn.Module):
    def __init__(self, out_dim):
        super().__init__()
        self.l1 = nn.Linear(784, HIDDEN); self.l2 = nn.Linear(HIDDEN, HIDDEN); self.l3 = nn.Linear(HIDDEN, out_dim)
    def forward(self, x):   # tanh keeps the loss smooth, so second derivatives exist
        return self.l3(torch.tanh(self.l2(torch.tanh(self.l1(x)))))

def setup(scenario):
    # task: 10 outputs, each task scored on its own 2 (mild forgetting)
    # domain: all tasks share the same 2 outputs (strong forgetting)
    global SCENARIO, model, P_NAMES, P_SHAPES, P_SIZES, NPARAM
    SCENARIO = scenario
    model = MLP(2 if scenario == "domain" else 10).to(dev)
    P_NAMES = [n for n, _ in model.named_parameters()]
    P_SHAPES = [p.shape for _, p in model.named_parameters()]
    P_SIZES = [p.numel() for _, p in model.named_parameters()]
    NPARAM = sum(P_SIZES)

def unflat(th):
    out, i = {}, 0
    for n, s, k in zip(P_NAMES, P_SHAPES, P_SIZES):
        out[n] = th[i:i + k].view(s); i += k
    return out

def init(seed):
    torch.manual_seed(seed)
    m = MLP(2 if SCENARIO == "domain" else 10).to(dev)
    return torch.cat([p.detach().flatten() for p in m.parameters()])

def logits(th, X, cls):
    z = functional_call(model, unflat(th), (X,))
    return z if SCENARIO == "domain" else z[:, cls]

def task_loss(th, X, y, cls):
    return F.cross_entropy(logits(th, X, cls), (y == cls[1]).long())

def grad(th, t, X=None, y=None):
    d = data[t]; X = d["Xest"] if X is None else X; y = d["yest"] if y is None else y
    th = th.detach().requires_grad_(True)
    return torch.autograd.grad(task_loss(th, X, y, d["cls"]), th)[0].detach()

def hvp(th, t, v):
    # exact Hessian-vector product (double backprop) on the task's estimation images
    d = data[t]; th = th.detach().requires_grad_(True)
    g = torch.autograd.grad(task_loss(th, d["Xest"], d["yest"], d["cls"]), th, create_graph=True)[0]
    return torch.autograd.grad(g @ v.detach(), th)[0].detach()

def bracket(th, a, b):
    ga, gb = grad(th, a), grad(th, b)
    return hvp(th, b, ga) - hvp(th, a, gb), ga, gb

@torch.no_grad()
def evaluate(th, t, split="te"):
    d = data[t]; X, y = d["X" + split], d["y" + split]
    z = logits(th, X, d["cls"]); yi = (y == d["cls"][1]).long()
    return float((z.argmax(1) == yi).float().mean()), float(F.cross_entropy(z, yi))

def train(th, t, seed, record=None, record_every=5):
    d = data[t]; th = th.detach().clone()
    gen = torch.Generator().manual_seed(seed * 1000 + t)
    n = len(d["ytr"]); step = 0
    for ep in range(EPOCHS):
        idx = torch.randperm(n, generator=gen).to(dev)
        for i in range(0, n, BATCH):
            bi = idx[i:i + BATCH]
            if record is not None and step % record_every == 0: record(step, th)
            th = th - LR * grad(th, t, d["Xtr"][bi], d["ytr"][bi])   # plain SGD
            step += 1
    if record is not None: record(step, th)
    return th
print("ready")

ready


## Cell 5: quick checks (about a minute)
Three checks before the real run:
1. **Symmetry:** the two numbers should be almost equal.
2. **Accuracy:** each task trained alone should reach above 0.97.
3. **Formula check:** the ratio should get close to 1.00 as the learning rate gets smaller (like 0.99 to 1.00 in the pilot).

If any of these fail, stop and send Claude the output.

In [6]:
setup("task")
th = init(99)
u, v = torch.randn(NPARAM, device=dev), torch.randn(NPARAM, device=dev)
print(f"1. symmetry: {float(u @ hvp(th, 0, v)):.5f}  vs  {float(v @ hvp(th, 0, u)):.5f}")

t0 = time.time(); thA = train(init(0), 0, 0)
print(f"2. task 0 accuracy: {evaluate(thA, 0)[0]:.4f}  (one task took {time.time() - t0:.1f}s)")

th = init(0); br, ga, gb = bracket(th, 0, 1)
for eta in [0.3, 0.1, 0.03, 0.01, 0.003]:
    t1 = th - eta * ga; tAB = t1 - eta * grad(t1, 1)
    t2 = th - eta * gb; tBA = t2 - eta * grad(t2, 0)
    d = tAB - tBA; pr = eta ** 2 * br
    print(f"3. lr {eta:<6} ratio {float(d.norm() / pr.norm()):.3f}  direction match {float(F.cosine_similarity(d, pr, dim=0)):.3f}")

1. symmetry: -125.12407  vs  -125.12408
2. task 0 accuracy: 0.9991  (one task took 1.6s)
3. lr 0.3    ratio 1.001  direction match 0.907
3. lr 0.1    ratio 1.007  direction match 0.988
3. lr 0.03   ratio 1.003  direction match 0.999
3. lr 0.01   ratio 1.001  direction match 1.000
3. lr 0.003  ratio 1.001  direction match 0.999


## Cell 6: the experiment function
Defines the full run. **Should print:** `defined`.

In [7]:
def run(scenario, seeds=SEEDS):
    setup(scenario)
    out = os.path.join(RESULTS_DIR, scenario); os.makedirs(out, exist_ok=True)
    pairs = [(a, b) for a in range(5) for b in range(5) if a != b]
    rows, t0 = [], time.time()
    for seed in range(seeds):
        th0 = init(seed)
        th_task = {t: train(th0, t, seed) for t in range(5)}
        base = {t: evaluate(th_task[t], t) for t in range(5)}
        base_tr = {t: evaluate(th_task[t], t, "est")[1] for t in range(5)}
        th_pair, early = {}, {}
        for a, b in pairs:
            log = []
            def rec(step, th_, a=a, b=b, log=log):
                if step in (5, 10, 15, 20):
                    br_, _, _ = bracket(th_, a, b)
                    log.append((float(br_.norm()), evaluate(th_, a, "est")[1]))
            th_pair[(a, b)] = train(th_task[a], b, seed, record=rec)
            early[(a, b)] = (float(np.mean([x[0] for x in log])), log[-1][1] - base_tr[a])
        for a, b in pairs:
            thA, thAB, thBA = th_task[a], th_pair[(a, b)], th_pair[(b, a)]
            accA0, lossA0 = base[a]; accA1, lossA1 = evaluate(thAB, a)
            ga, gb = grad(thA, a), grad(thA, b)
            HBgA, HAgB = hvp(thA, b, ga), hvp(thA, a, gb)
            with torch.no_grad():
                Xu = torch.cat([data[a]["Xte"], data[b]["Xte"]])
                pa = functional_call(model, unflat(thAB), (Xu,)).argmax(1)
                pb = functional_call(model, unflat(thBA), (Xu,)).argmax(1)
            rows.append(dict(seed=seed, A=a, B=b, acc_A_before=accA0, acc_A_after=accA1,
                             forgetting=accA0 - accA1, loss_increase=lossA1 - lossA0,
                             bracket_norm=float((HBgA - HAgB).norm()), bracket_early=early[(a, b)][0],
                             curvature_term=float(gb @ HAgB), grad_cos=float(F.cosine_similarity(ga, gb, dim=0)),
                             gB_norm=float(gb.norm()), early_loss_rise_A=early[(a, b)][1],
                             order_gap_predictions=float((pa != pb).float().mean())))
        print(f"  seed {seed} done after {time.time() - t0:.0f}s | mean task accuracy {np.mean([base[t][0] for t in range(5)]):.4f}")
    with open(os.path.join(out, "results.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
    col = lambda k: np.array([r[k] for r in rows])
    def resid(yv, x):
        x1 = np.c_[np.ones_like(x), x]; return yv - x1 @ np.linalg.lstsq(x1, yv, rcond=None)[0]
    S = {"scenario": scenario, "dataset": "MNIST", "seeds": seeds, "n_pairs": len(rows),
         "mean_task_accuracy": float(col("acc_A_before").mean()),
         "forgetting_mean": float(col("forgetting").mean()), "forgetting_median": float(np.median(col("forgetting"))),
         "forgetting_max": float(col("forgetting").max()), "rho": {}}
    for k in ["bracket_early", "bracket_norm", "curvature_term", "gB_norm", "grad_cos", "early_loss_rise_A"]:
        r_, p_ = spearmanr(col(k), col("loss_increase")); S["rho"][k] = {"rho": float(r_), "p": float(p_)}
    rl = rankdata(col("loss_increase"))
    for k, c in [("bracket_early", "early_loss_rise_A"), ("bracket_early", "gB_norm"), ("bracket_norm", "curvature_term")]:
        rk, rc = rankdata(col(k)), rankdata(col(c))
        S[f"partial_{k}_given_{c}"] = float(np.corrcoef(resid(rk, rc), resid(rl, rc))[0, 1])
    # forgetting curve for the most-forgotten pair
    mf = {}
    for r_ in rows: mf.setdefault((r_["A"], r_["B"]), []).append(r_["forgetting"])
    a, b = max(mf, key=lambda k: np.mean(mf[k])); S["curve_pair"] = [a, b]
    curve = []
    def rec2(step, th_):
        br_, _, _ = bracket(th_, a, b)
        curve.append(dict(step=step, acc_A=evaluate(th_, a)[0], acc_B=evaluate(th_, b)[0], bracket_norm=float(br_.norm())))
    train(train(init(0), a, 0), b, 0, record=rec2, record_every=10)
    json.dump(curve, open(os.path.join(out, "curve.json"), "w"))
    json.dump(S, open(os.path.join(out, "summary.json"), "w"), indent=2)
    print(f"finished {scenario} in {time.time() - t0:.0f}s")
    return S, rows, curve
print("defined")

defined


## Cell 7: run the MILD setting (task)
**Should print:** one line per seed with mean task accuracy above 0.97, then `finished task`.
Note how long seed 0 takes: that tells you how long the whole run will take.

In [8]:
S_task, rows_task, curve_task = run("task")

  seed 0 done after 40s | mean task accuracy 0.9886
  seed 1 done after 80s | mean task accuracy 0.9889
  seed 2 done after 120s | mean task accuracy 0.9891
  seed 3 done after 158s | mean task accuracy 0.9902
  seed 4 done after 198s | mean task accuracy 0.9894
finished task in 205s


## Cell 8: run the STRONG setting (domain)
Same as above, with shared outputs, so forgetting is much bigger.

In [9]:
S_dom, rows_dom, curve_dom = run("domain")

  seed 0 done after 36s | mean task accuracy 0.9882
  seed 1 done after 70s | mean task accuracy 0.9891
  seed 2 done after 105s | mean task accuracy 0.9902
  seed 3 done after 141s | mean task accuracy 0.9899
  seed 4 done after 174s | mean task accuracy 0.9892
finished domain in 180s


## Cell 9: the results table
Each number is the rank correlation with forgetting (1 = perfect, 0 = no link). Compare with the pilot.

In [10]:
labels = {"bracket_early": "Lie bracket, first 20 steps", "bracket_norm": "Lie bracket at the switch",
          "curvature_term": "EWC-style curvature", "gB_norm": "Size of task B's push",
          "grad_cos": "Gradient alignment", "early_loss_rise_A": "Simple check (old loss after 20 steps)"}
pilot = {"bracket_early": (0.82, 0.73), "bracket_norm": (0.59, 0.21), "curvature_term": (0.43, 0.16),
         "gB_norm": (0.05, 0.27), "grad_cos": (0.12, -0.08), "early_loss_rise_A": (0.65, 0.76)}
print(f"{'signal':40s} {'MILD':>6s} {'STRONG':>7s}   pilot (mild / strong)")
for k, name in labels.items():
    print(f"{name:40s} {S_task['rho'][k]['rho']:6.2f} {S_dom['rho'][k]['rho']:7.2f}   {pilot[k][0]:.2f} / {pilot[k][1]:.2f}")
for S in (S_task, S_dom):
    print(f"\n{S['scenario']}: task accuracy {S['mean_task_accuracy']:.3f} | forgetting mean {S['forgetting_mean']:.3f}, "
          f"median {S['forgetting_median']:.3f}, max {S['forgetting_max']:.3f}")
    print("  partial (bracket early, after removing the simple check):", round(S["partial_bracket_early_given_early_loss_rise_A"], 2))

signal                                     MILD  STRONG   pilot (mild / strong)
Lie bracket, first 20 steps                0.82    0.69   0.82 / 0.73
Lie bracket at the switch                  0.56    0.28   0.59 / 0.21
EWC-style curvature                        0.48    0.27   0.43 / 0.16
Size of task B's push                      0.36    0.44   0.05 / 0.27
Gradient alignment                         0.05   -0.06   0.12 / -0.08
Simple check (old loss after 20 steps)     0.87    0.86   0.65 / 0.76

task: task accuracy 0.989 | forgetting mean 0.013, median 0.008, max 0.071
  partial (bracket early, after removing the simple check): 0.52

domain: task accuracy 0.989 | forgetting mean 0.350, median 0.301, max 0.852
  partial (bracket early, after removing the simple check): -0.05


## Cell 10: figures
Saves three PNG charts to the results folder (same style as the pilot).

In [11]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"font.family": "Calibri", "font.size": 12, "axes.spines.top": False, "axes.spines.right": False})
GREEN, AMBER, INK, MUTED, GRID = "#1F8A70", "#B8651C", "#1F2624", "#5B6664", "#E3E8E6"

# 1) predictors
ks = list(labels); y = np.arange(len(ks))[::-1]; h = 0.36
fig, ax = plt.subplots(figsize=(8.2, 5.0), dpi=200)
vt = [S_task["rho"][k]["rho"] for k in ks]; vd = [S_dom["rho"][k]["rho"] for k in ks]
ax.barh(y + h / 2, vt, height=h, color=GREEN, label="Mild forgetting (separate outputs)")
ax.barh(y - h / 2, vd, height=h, color=AMBER, label="Strong forgetting (shared outputs)")
for yy, v in list(zip(y + h / 2, vt)) + list(zip(y - h / 2, vd)):
    ax.text(v + (0.015 if v >= 0 else -0.015), yy, f"{v:.2f}", va="center", ha="left" if v >= 0 else "right", fontsize=10, color=INK)
ax.set_yticks(y); ax.set_yticklabels([labels[k] for k in ks]); ax.axvline(0, color="#9AA6A3", lw=0.8)
ax.set_xlim(-0.2, 1.0); ax.set_xlabel("Rank correlation with forgetting (Spearman ρ; higher = better)")
ax.grid(axis="x", color=GRID); ax.set_axisbelow(True)
ax.legend(loc="lower left", bbox_to_anchor=(-0.02, 1.0), ncol=2, frameon=False, fontsize=10)
ax.set_title("MNIST: how well each signal predicts forgetting", loc="left", fontsize=13, weight="bold", pad=28)
fig.tight_layout(); fig.savefig(os.path.join(RESULTS_DIR, "mnist_predictors.png"), facecolor="white"); plt.close(fig)

# 2) forgetting curve (strong setting)
a, b = S_dom["curve_pair"]; st = [c["step"] for c in curve_dom]
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7.2, 5.2), dpi=200, sharex=True, gridspec_kw={"height_ratios": [1.4, 1]})
ax1.plot(st, [100 * c["acc_A"] for c in curve_dom], color=GREEN, lw=2.2, label=f"Old task A ({NAMES[a]})")
ax1.plot(st, [100 * c["acc_B"] for c in curve_dom], color=AMBER, lw=2.2, ls="--", label=f"New task B ({NAMES[b]})")
ax1.set_ylabel("Test accuracy (%)"); ax1.set_ylim(0, 105); ax1.legend(frameon=False, loc="lower right"); ax1.grid(axis="y", color=GRID)
ax1.set_title("MNIST: training on B erases A, and the bracket rises", loc="left", fontsize=13, weight="bold")
ax2.plot(st, [c["bracket_norm"] for c in curve_dom], color=GREEN, lw=2.2)
ax2.set_ylabel("Lie bracket size"); ax2.set_xlabel("Training steps on task B"); ax2.grid(axis="y", color=GRID)
fig.tight_layout(); fig.savefig(os.path.join(RESULTS_DIR, "mnist_forgetting_curve.png"), facecolor="white"); plt.close(fig)

# 3) scatter
fig, axs = plt.subplots(1, 2, figsize=(8.6, 3.8), dpi=200)
for ax, R, S, col, title in [(axs[0], rows_task, S_task, GREEN, "Mild"), (axs[1], rows_dom, S_dom, AMBER, "Strong")]:
    ax.scatter([r["bracket_early"] for r in R], [r["loss_increase"] for r in R], s=26, color=col, alpha=0.75, edgecolor="white", linewidth=0.6)
    ax.set_xscale("log"); ax.grid(color=GRID); ax.set_xlabel("Lie bracket, first 20 steps (log)")
    ax.set_title(f"{title} forgetting: ρ = {S['rho']['bracket_early']['rho']:.2f}", loc="left", fontsize=12, weight="bold")
axs[0].set_ylabel("Forgetting (rise in test loss)")
fig.tight_layout(); fig.savefig(os.path.join(RESULTS_DIR, "mnist_scatter.png"), facecolor="white"); plt.close(fig)
print("saved figures to", RESULTS_DIR)

saved figures to F:\forgetting\results


## Done
Send Claude: the output of **Cell 5**, **Cell 9**, and the two `summary.json` files (in `results\task` and `results\domain`), plus the three PNGs.